# Item

> File and folder item rendering components for the file browser.

In [ ]:
#| default_exp components.item

In [ ]:
#| export
from typing import Any, Dict, Optional

from fasthtml.common import Div, Span, Button, Tr, Td, Input

# DaisyUI components
from cjm_fasthtml_daisyui.components.actions.button import btn, btn_colors, btn_sizes, btn_styles
from cjm_fasthtml_daisyui.components.data_input.checkbox import checkbox, checkbox_colors, checkbox_sizes
from cjm_fasthtml_daisyui.utilities.semantic_colors import bg_dui, text_dui, border_dui

# Tailwind utilities
from cjm_fasthtml_tailwind.utilities.spacing import p, m
from cjm_fasthtml_tailwind.utilities.sizing import w, h, min_w
from cjm_fasthtml_tailwind.utilities.typography import (
    font_size, font_weight, font_family, truncate, text_nowrap
)
from cjm_fasthtml_tailwind.utilities.borders import border, rounded
from cjm_fasthtml_tailwind.utilities.flexbox_and_grid import (
    flex_display, flex_direction, justify, items, gap, grow, shrink
)
from cjm_fasthtml_tailwind.utilities.interactivity import cursor
from cjm_fasthtml_tailwind.core.base import combine_classes

# Lucide icons
from cjm_fasthtml_lucide_icons.factory import lucide_icon

# Local imports
from cjm_file_discovery.core.models import FileInfo, FileType
from cjm_fasthtml_file_browser.core.config import FileBrowserConfig, ViewMode

## Icon Mapping

Maps file types to Lucide icon names for consistent iconography.

In [ ]:
#| export
FILE_TYPE_ICONS: Dict[FileType, str] = {
    FileType.AUDIO: "file-music",
    FileType.VIDEO: "file-video-camera",
    FileType.IMAGE: "image",
    FileType.DOCUMENT: "file-text",
    FileType.CODE: "file-code",
    FileType.DATA: "database",
    FileType.ARCHIVE: "archive",
    FileType.OTHER: "file",
}

BROWSER_ICONS: Dict[str, str] = {
    "folder": "folder",
    "folder_open": "folder-open",
    "parent": "arrow-up",
    "home": "house",
    "refresh": "refresh-cw",
    "list_view": "list",
    "grid_view": "grid-3x3",
    "sort_asc": "arrow-up-narrow-wide",
    "sort_desc": "arrow-down-wide-narrow",
    "check": "check",
    "x": "x",
}

In [ ]:
# Test icon mapping
assert FILE_TYPE_ICONS[FileType.CODE] == "file-code"
assert FILE_TYPE_ICONS[FileType.AUDIO] == "file-music"
assert BROWSER_ICONS["folder"] == "folder"
print("Icon mapping tests passed!")

Icon mapping tests passed!


## Icon Rendering

In [ ]:
#| export
def _get_file_icon(
    file_info: FileInfo,  # File to get icon for
    size: int = 4         # Icon size (Tailwind scale)
) -> Any:  # Lucide icon component
    """Get the appropriate icon for a file based on its type."""
    if file_info.is_directory:
        icon_name = BROWSER_ICONS["folder"]
        color_cls = str(text_dui.warning)
    else:
        icon_name = FILE_TYPE_ICONS.get(file_info.file_type, "file")
        # Color based on file type
        color_map = {
            FileType.CODE: str(text_dui.info),
            FileType.DATA: str(text_dui.primary),
            FileType.IMAGE: str(text_dui.success),
            FileType.AUDIO: str(text_dui.secondary),
            FileType.VIDEO: str(text_dui.accent),
            FileType.DOCUMENT: str(text_dui.base_content),
            FileType.ARCHIVE: str(text_dui.warning),
        }
        color_cls = color_map.get(file_info.file_type, str(text_dui.base_content))
    
    return lucide_icon(icon_name, size=size, cls=color_cls)

In [ ]:
from fasthtml.common import to_xml

# Test icon rendering - verify icons are generated without errors
folder = FileInfo(name="test", path="/test", is_directory=True)
icon = _get_file_icon(folder)
html = to_xml(icon)
assert "<svg" in html.lower()  # SVG element present
assert "text-warning" in html  # Folder color class

py_file = FileInfo(name="test.py", path="/test.py", is_directory=False, file_type=FileType.CODE)
icon = _get_file_icon(py_file)
html = to_xml(icon)
assert "<svg" in html.lower()
assert "text-info" in html  # Code file color class

print("Icon rendering tests passed!")

Icon rendering tests passed!


## List View Item

Renders a file or folder as a table row for list view.

In [ ]:
#| export
def render_list_item(
    file_info: FileInfo,                      # File to render
    config: FileBrowserConfig,                # Browser configuration
    is_selected: bool = False,                # Whether item is selected
    item_id: Optional[str] = None,            # HTML ID for the item
    navigate_url: Optional[str] = None,       # URL for directory navigation
    select_url: Optional[str] = None,         # URL for file selection
    hx_target: Optional[str] = None,          # HTMX target for swaps
) -> Any:  # Table row component
    """Render a file/folder as a list view row."""
    can_select = config.can_select(file_info)
    
    # Row styling
    row_cls = combine_classes(
        "hover",  # DaisyUI table hover
        cursor.pointer if (file_info.is_directory or can_select) else None,
        bg_dui.primary.opacity(10) if is_selected else None
    )
    
    # Build row attributes
    row_attrs = {"cls": row_cls}
    if item_id:
        row_attrs["id"] = item_id
        row_attrs["data-path"] = file_info.path
    
    # Click behavior: navigate for folders, select for files
    if file_info.is_directory and navigate_url:
        row_attrs["hx_post"] = navigate_url
        row_attrs["hx_vals"] = f'{{"path": "{file_info.path}"}}'
        if hx_target:
            row_attrs["hx_target"] = hx_target
        row_attrs["hx_swap"] = "outerHTML"
    elif can_select and select_url:
        row_attrs["hx_post"] = select_url
        row_attrs["hx_vals"] = f'{{"path": "{file_info.path}"}}'
        if hx_target:
            row_attrs["hx_target"] = hx_target
        row_attrs["hx_swap"] = "outerHTML"
    
    # Build cells based on configured columns
    cells = []
    
    # Selection checkbox (if selection enabled)
    if config.selection_mode.value != "none" and can_select:
        # Build checkbox attributes for HTMX selection
        checkbox_attrs = {
            "type": "checkbox",
            "checked": is_selected,
            "cls": combine_classes(checkbox, checkbox_colors.primary, checkbox_sizes.sm),
            "onclick": "event.stopPropagation()",  # Don't trigger row click
        }
        # Add HTMX attributes to make checkbox directly clickable
        if select_url:
            checkbox_attrs["hx_post"] = select_url
            checkbox_attrs["hx_vals"] = f'{{"path": "{file_info.path}"}}'
            if hx_target:
                checkbox_attrs["hx_target"] = hx_target
            checkbox_attrs["hx_swap"] = "outerHTML"
        
        cells.append(Td(
            Input(**checkbox_attrs),
            cls=str(w(8))
        ))
    
    # Name cell (always present)
    name_content = Div(
        Span(_get_file_icon(file_info, size=4), cls=str(m.r(2))),
        Span(file_info.name, cls=combine_classes(truncate, grow())),
        cls=combine_classes(flex_display, items.center, min_w._0)
    )
    cells.append(Td(name_content))
    
    # Size cell
    from cjm_fasthtml_file_browser.core.config import FileColumn
    if FileColumn.SIZE in config.view.columns:
        size_text = file_info.size_str if not file_info.is_directory else "—"
        cells.append(Td(
            size_text,
            cls=combine_classes(text_nowrap, text_dui.base_content.opacity(70), font_size.sm)
        ))
    
    # Modified cell
    if FileColumn.MODIFIED in config.view.columns:
        cells.append(Td(
            file_info.modified_str,
            cls=combine_classes(text_nowrap, text_dui.base_content.opacity(70), font_size.sm)
        ))
    
    # Type cell
    if FileColumn.TYPE in config.view.columns:
        type_text = "Folder" if file_info.is_directory else (file_info.extension or "File").upper()
        cells.append(Td(
            type_text,
            cls=combine_classes(text_nowrap, text_dui.base_content.opacity(70), font_size.sm)
        ))
    
    return Tr(*cells, **row_attrs)

In [ ]:
# Test list view item
config = FileBrowserConfig()

# Test folder rendering
folder = FileInfo(name="Documents", path="/home/user/Documents", is_directory=True)
row = render_list_item(folder, config, item_id="item-0", navigate_url="/navigate")
html = to_xml(row)
assert "Documents" in html
assert "item-0" in html
assert "<svg" in html.lower()  # Icon is rendered
assert "hx-post" in html  # HTMX navigation

# Test file rendering with selection
file = FileInfo(
    name="test.py", 
    path="/home/user/test.py", 
    is_directory=False,
    size=1024,
    file_type=FileType.CODE,
    extension="py"
)
row = render_list_item(file, config, is_selected=True, select_url="/select")
html = to_xml(row)
assert "test.py" in html
assert "checked" in html  # Selected checkbox

print("List view item tests passed!")

List view item tests passed!


## Grid View Item

Renders a file or folder as a card for grid view.

In [ ]:
#| export
def render_grid_item(
    file_info: FileInfo,                      # File to render
    config: FileBrowserConfig,                # Browser configuration
    is_selected: bool = False,                # Whether item is selected
    item_id: Optional[str] = None,            # HTML ID for the item
    navigate_url: Optional[str] = None,       # URL for directory navigation
    select_url: Optional[str] = None,         # URL for file selection
    hx_target: Optional[str] = None,          # HTMX target for swaps
) -> Any:  # Grid card component
    """Render a file/folder as a grid view card."""
    can_select = config.can_select(file_info)
    
    # Card styling (include "relative" for absolute positioning of selection indicator)
    card_cls = combine_classes(
        "relative",  # For absolute positioning of selection indicator
        flex_display, flex_direction.col, items.center, justify.center,
        p(3), rounded.lg,
        bg_dui.base_200 if not is_selected else bg_dui.primary.opacity(20),
        border(), border_dui.base_300 if not is_selected else border_dui.primary,
        cursor.pointer if (file_info.is_directory or can_select) else None,
        "hover:bg-base-300" if not is_selected else None,  # Hover effect
    )
    
    # Build card attributes
    card_attrs = {"cls": card_cls}
    if item_id:
        card_attrs["id"] = item_id
        card_attrs["data-path"] = file_info.path
    
    # Click behavior
    if file_info.is_directory and navigate_url:
        card_attrs["hx_post"] = navigate_url
        card_attrs["hx_vals"] = f'{{"path": "{file_info.path}"}}'
        if hx_target:
            card_attrs["hx_target"] = hx_target
        card_attrs["hx_swap"] = "outerHTML"
    elif can_select and select_url:
        card_attrs["hx_post"] = select_url
        card_attrs["hx_vals"] = f'{{"path": "{file_info.path}"}}'
        if hx_target:
            card_attrs["hx_target"] = hx_target
        card_attrs["hx_swap"] = "outerHTML"
    
    # Selection indicator
    selection_indicator = None
    if config.selection_mode.value != "none" and can_select:
        # Build indicator attributes
        indicator_attrs = {
            "cls": combine_classes(
                w(5), h(5), rounded.full,
                bg_dui.primary if is_selected else bg_dui.base_300,
                flex_display, items.center, justify.center,
                "absolute top-2 right-2",
                cursor.pointer
            ),
            "onclick": "event.stopPropagation()",  # Don't trigger card click
        }
        # Add HTMX attributes to make indicator directly clickable
        if select_url:
            indicator_attrs["hx_post"] = select_url
            indicator_attrs["hx_vals"] = f'{{"path": "{file_info.path}"}}'
            if hx_target:
                indicator_attrs["hx_target"] = hx_target
            indicator_attrs["hx_swap"] = "outerHTML"
        
        selection_indicator = Div(
            lucide_icon("check", size=3, cls=str(text_dui.primary_content)) if is_selected else None,
            **indicator_attrs
        )
    
    return Div(
        # Selection indicator (positioned absolutely)
        selection_indicator,
        # Large icon
        Div(
            _get_file_icon(file_info, size=8),
            cls=str(m.b(2))
        ),
        # File name
        Span(
            file_info.name,
            cls=combine_classes(
                font_size.sm, truncate, w.full, 
                "text-center", font_weight.medium
            ),
            title=file_info.name  # Full name on hover
        ),
        # File size (for files only)
        Span(
            file_info.size_str if not file_info.is_directory else "",
            cls=combine_classes(font_size.xs, text_dui.base_content.opacity(60))
        ) if not file_info.is_directory else None,
        **card_attrs
    )

In [ ]:
# Test grid view item
config = FileBrowserConfig()

# Test folder rendering
folder = FileInfo(name="Documents", path="/home/user/Documents", is_directory=True)
card = render_grid_item(folder, config, item_id="grid-0", navigate_url="/navigate")
html = to_xml(card)
assert "Documents" in html
assert "grid-0" in html

# Test file rendering with selection
file = FileInfo(
    name="image.png", 
    path="/home/user/image.png", 
    is_directory=False,
    size=2048,
    file_type=FileType.IMAGE,
    extension="png"
)
card = render_grid_item(file, config, is_selected=True, select_url="/select")
html = to_xml(card)
assert "image.png" in html
assert "2.0 KB" in html or "2 KB" in html  # Size displayed

print("Grid view item tests passed!")

Grid view item tests passed!


## Parent Navigation Item

Special item for navigating to parent directory.

In [ ]:
#| export
def render_parent_item(
    parent_path: str,                         # Parent directory path
    view_mode: ViewMode,                      # Current view mode
    navigate_url: str,                        # URL for navigation
    hx_target: Optional[str] = None,          # HTMX target
) -> Any:  # Parent navigation component
    """Render the parent directory navigation item."""
    common_attrs = {
        "hx_post": navigate_url,
        "hx_vals": f'{{"path": "{parent_path}"}}',
        "hx_swap": "outerHTML",
    }
    if hx_target:
        common_attrs["hx_target"] = hx_target
    
    icon = lucide_icon(BROWSER_ICONS["parent"], size=4, cls=str(text_dui.base_content))
    
    if view_mode == ViewMode.LIST:
        return Tr(
            Td(
                Div(
                    Span(icon, cls=str(m.r(2))),
                    Span("..", cls=str(text_dui.base_content.opacity(70))),
                    cls=combine_classes(flex_display, items.center)
                ),
                colspan="99"  # Span all columns
            ),
            cls=combine_classes("hover", cursor.pointer),
            **common_attrs
        )
    else:  # Grid view
        return Div(
            Div(icon, cls=str(m.b(2))),
            Span("..", cls=combine_classes(font_size.sm, text_dui.base_content.opacity(70))),
            cls=combine_classes(
                flex_display, flex_direction.col, items.center, justify.center,
                p(3), rounded.lg, bg_dui.base_200, border(), border_dui.base_300,
                cursor.pointer, "hover:bg-base-300"
            ),
            **common_attrs
        )

In [ ]:
# Test parent navigation item
# List view
parent_list = render_parent_item("/home", ViewMode.LIST, "/navigate")
html = to_xml(parent_list)
assert ".." in html
assert "/navigate" in html

# Grid view
parent_grid = render_parent_item("/home", ViewMode.GRID, "/navigate")
html = to_xml(parent_grid)
assert ".." in html

print("Parent navigation item tests passed!")

Parent navigation item tests passed!


## Unified Render Function

Single function that dispatches to list or grid rendering based on view mode.

In [ ]:
#| export
def render_item(
    file_info: FileInfo,                      # File to render
    config: FileBrowserConfig,                # Browser configuration
    view_mode: ViewMode,                      # Current view mode
    is_selected: bool = False,                # Whether item is selected
    item_id: Optional[str] = None,            # HTML ID for the item
    navigate_url: Optional[str] = None,       # URL for directory navigation
    select_url: Optional[str] = None,         # URL for file selection
    hx_target: Optional[str] = None,          # HTMX target for swaps
) -> Any:  # Item component (row or card)
    """Render a file/folder item based on view mode."""
    if view_mode == ViewMode.LIST:
        return render_list_item(
            file_info, config, is_selected, item_id,
            navigate_url, select_url, hx_target
        )
    else:
        return render_grid_item(
            file_info, config, is_selected, item_id,
            navigate_url, select_url, hx_target
        )

In [ ]:
# Test unified render function
config = FileBrowserConfig()
file = FileInfo(name="test.txt", path="/test.txt", is_directory=False)

# List mode should return Tr
list_item = render_item(file, config, ViewMode.LIST)
assert "<tr" in to_xml(list_item).lower()

# Grid mode should return Div
grid_item = render_item(file, config, ViewMode.GRID)
assert "<div" in to_xml(grid_item).lower()

print("Unified render function tests passed!")

Unified render function tests passed!


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()